In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from benchmark_data_leakage.utils import find_repo_root

repo_root = find_repo_root()
data_dir = repo_root / "data"
out_dir = data_dir / "out"
fig_dir = repo_root / "figures"
fig_dir.mkdir(exist_ok=True)

In [ ]:
FEPp_data_points = [
    # These are from the Boltz-2 and IsoDDE papers
    ("IsoDDE", 0.85, None, None),
    ("FEP+", 0.78, None, None),
    ("OpenFE", 0.66, None, None),
    ("Boltz-2", 0.66, None, None),
    # These were extracted with a ligand only model, and can be reproduced by following:
    # https://github.com/bamattsson/ntab/blob/17396ec9e34deba0f65910fd57571c8671fc4aef/reproduce_results_on_fep4/00_instructions.md
    ("FP + mol prop", 0.6631, 0.5761, 0.7807),  # r=0.6631  (SE=0.0537, 95% CI=[0.5761, 0.7807])
    ("Chemprop", 0.6158, 0.4730, 0.8007),  # r=0.6158  (SE=0.0938, 95% CI=[0.4730, 0.8007])
    # ("FP only", 0.5290, 0.3565, 0.7406),  # r=0.5290  (SE=0.0951, 95% CI=[0.3565, 0.7406])
    ("mol prop only", 0.4855, 0.0807, 0.6504),  # r=0.4855  (SE=0.1488, 95% CI=[0.0807, 0.6504])
    # Data size: n_rows=87 n_assays=4
]

OpenFE_data_points = [
    # These are from the Boltz-2 and IsoDDE papers
    ("IsoDDE", 0.73, None, None),
    ("FEP+", 0.72, None, None),
    ("OpenFE", 0.63, None, None),
    ("Boltz-2", 0.62, None, None),
    # These were extracted with a ligand only model, and can be reproduced by following:
    # https://github.com/bamattsson/ntab/blob/17396ec9e34deba0f65910fd57571c8671fc4aef/reproduce_results_on_fep4/00_instructions.md
    ("Chemprop", 0.3551, 0.2472, 0.4557),  # r=0.3551  (SE=0.0549, 95% CI=[0.2472, 0.4557])
    # ("FP only", 0.3335, 0.2193, 0.4491),  # r=0.3335  (SE=0.0572, 95% CI=[0.2193, 0.4491])
    ("mol prop only", 0.2877, 0.1523, 0.4074),  # r=0.2877  (SE=0.0649, 95% CI=[0.1523, 0.4074])
    ("FP + mol prop", 0.2692, 0.1741, 0.3774),  # r=0.2692  (SE=0.0517, 95% CI=[0.1741, 0.3774])
    # Data size: n_rows=873 n_assays=58
]

In [ ]:
C_PHYSICS  = "#93cfe0"   # pale teal
C_AI_MODEL = "#e8a8b0"   # pale rose
C_BASELINE = "#d8d898"   # pale olive

colours = [C_AI_MODEL, C_PHYSICS, C_PHYSICS, C_AI_MODEL, C_BASELINE, C_BASELINE, C_BASELINE, C_BASELINE]

fig, axes = plt.subplots(1, 2, figsize=(10.0, 5.2), sharey=True,
                         gridspec_kw={"wspace": 0.05})

for ax, data_points, title in zip(axes, [FEPp_data_points, OpenFE_data_points],
                                   ["FEP+ 4", "OpenFE"]):
    labels = [m for m, _, _, _ in data_points]
    values = [v for _, v, _, _ in data_points]
    ci_los = [lo for _, _, lo, _ in data_points]
    ci_his = [hi for _, _, _, hi in data_points]
    x_pos = np.arange(len(labels)) * 0.8

    # Build asymmetric yerr; use 0 where CI is absent
    yerr_lo = [v - lo if lo is not None else 0 for v, lo in zip(values, ci_los)]
    yerr_hi = [hi - v if hi is not None else 0 for v, hi in zip(values, ci_his)]
    yerr = [yerr_lo, yerr_hi]

    bars = ax.bar(x_pos, values, color=colours, width=0.45, zorder=2,
                  edgecolor="#323232", linewidth=0.6,
                  yerr=yerr, capsize=3,
                  error_kw=dict(elinewidth=0.5, ecolor="#323232", capthick=0.8, zorder=5))

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, val / 2,
                f"{val:.2f}", ha="center", va="center", fontsize=9,
                rotation=90, color="black")

    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, fontsize=9, rotation=90, ha="center")
    ax.set_ylim(0, 1.05)
    ax.set_title(title, fontsize=11)

    ax.grid(visible=True, which="major", axis="y", linestyle=":", linewidth=0.5, color="#cccccc")
    ax.set_axisbelow(True)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.tick_params(axis="x", length=0, labelsize=9)
    ax.tick_params(axis="y", labelsize=9)

axes[0].set_ylabel("Mean Pearson correlation ($r$)", fontsize=11)
axes[1].tick_params(axis="y", labelleft=False)

legend_handles = [
    mpatches.Patch(color=C_PHYSICS,  label="Physics-based methods"),
    mpatches.Patch(color=C_AI_MODEL, label="Structure-based ML models"),
    mpatches.Patch(color=C_BASELINE, label="Ligand-only ML baselines"),
]
fig.subplots_adjust(bottom=0.27)
fig.legend(handles=legend_handles, fontsize=9, frameon=False,
           loc="lower center", ncol=3)

plt.savefig(fig_dir / "FEPp_w_baselines_barplot.png", dpi=300, bbox_inches="tight")
plt.show()